# Part 5 — does vocabulary size help retrieval? Can PCA retain it?
Complements `3_retrieval.ipynb`, with the same corpus (val split), relevance (motorbike) and the same 10 query images (the DINOv2 k-means queries
printed by that notebook, fixed here so this notebook needs no deep features). Hellinger histograms, cosine ranking. PCA is fitted on the train split only.
The `K=512` row must reproduce the `BoVW K512` row of `results/part5_image_queries.csv`.


In [ ]:
import glob, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")
from src import eval as ev

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "data" / "images").exists())
DATA = str(ROOT / "data"); RES = ROOT / "code" / "results"
table = ev.split_table(DATA); names = table.name.values
corpus = np.where((table.split == "val").values)[0]; rel = table["motorbike"].values[corpus]
QUERY_IDS = ['002795', '009004', '001450', '009756', '008676', '005414', '000685', '000987', '004882', '006177']
queries = np.array([np.where(names == q)[0][0] for q in QUERY_IDS])
y = rel * 2 - 1
print(f"corpus {len(corpus)}, relevant {rel.sum()}, queries {len(queries)}")

def enc_matrix(e):
    d = {}
    for c in ev.ALL_CATS:
        for sp in ("train", "val"):
            z = np.load(f"{DATA}/encodings/{e}/{c}_{sp}.npz"); d.update(zip(map(str, z["file_names"]), z["histograms"]))
    return np.stack([d[str(n)] for n in names])

def run(F, pca=0):
    X = ev.hellinger(F); t_pca = 0.0
    if pca:
        t = time.perf_counter(); p = PCA(pca, random_state=0).fit(X[(table.split == "train").values]); t_pca = time.perf_counter() - t
        X = ev.l2(p.transform(X))
    Xc, Xq = X[corpus], X[queries]
    t = time.perf_counter(); S = Xq @ Xc.T; _ = np.argsort(-S, axis=1); t = (time.perf_counter() - t) / len(Xq)
    pq = pd.DataFrame([dict(AP=ev.ap(y, s), **{f"P@{k}": ev.precision_at_k(y, s, k) for k in (5, 10, 20)},
                            **{f"R@{k}": ev.recall_at_k(y, s, k) for k in (10, 50, 100)}) for s in S])
    return dict(dim=X.shape[1], mAP=pq.AP.mean(), AP_std_over_queries=pq.AP.std(), **pq.drop(columns="AP").mean().to_dict(),
                rank_ms_per_query=1e3 * t, corpus_MB=Xc.astype(np.float32).nbytes / 2**20, pca_fit_s=t_pca)


In [ ]:
rows = []
for e in sorted({Path(p).name for p in glob.glob(f"{DATA}/encodings/bovw_K*_C0")}, key=lambda s: int(s.split("_K")[1].split("_")[0])):
    K = int(e.split("_K")[1].split("_")[0]); F = enc_matrix(e)
    for pca in [0] + [p for p in (128, 256) if p < 4 * K]:
        rows.append(dict(K=K, pca=pca, **run(F, pca)))
V = pd.DataFrame(rows); display(V.round(3)); V.to_csv(RES / "part5_vocabulary.csv", index=False)
ref = pd.read_csv(RES / "part5_image_queries.csv").set_index("method").loc["BoVW K512", "mAP"]
print("K=512 mAP here:", round(V[(V.K == 512) & (V.pca == 0)].mAP.iloc[0], 6), "| notebook 3:", round(ref, 6))
fig, ax = plt.subplots(figsize=(6, 4))
for p, m in ((0, "o-"), (128, "s--"), (256, "^--")):
    a = V[V.pca == p]; ax.plot(a.K, a.mAP, m, label="no PCA" if p == 0 else f"PCA {p}")
ax.set_xscale("log", base=2); ax.set_xlabel("K"); ax.set_ylabel("mAP over 10 queries"); ax.legend(); ax.grid(alpha=.3); ax.set_title("Motorbike retrieval: vocabulary size"); plt.tight_layout(); plt.show()
